In [1]:
from pymongo import MongoClient #mengimpor mongoclient dari library python

client = MongoClient("mongodb://localhost:27017/") # 

db = client["praktikum_bigdata"]

koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada: ", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada:  ['admin', 'config', 'local', 'praktikum_bigdata']


In [2]:
koleksi_produk.delete_many({})

koleksi_produk.insert_one({
    "nama_produk": "Kopi Susu",
    "kategori": "Minuman",
    "harga": 18000,
    "stok": 120,
    "tags": ["best-seller", "dingin", "panas", "hangat"]
})

koleksi_produk.insert_many([
    {"nama_produk": "Roti Bakar", "kategori": "Makanan", "harga": 3000, "stok": 10, "tags": ["smokey", "hangat"]},
    {"nama_produk": "Es Teh", "kategori": "Minuman", "harga": 10000, "stok": 25, "tags": ["manis", "Tidak"]},
    {"nama_produk": "Kentang Goreng", "kategori": "Makanan","harga": 200000, "stok": 5, "tags": ["asin"]},
     {"nama_produk": "Cappuccino", "kategori": "Minuman", "harga": 25000, "stok": 80, "tags": ["hangat", "best-seller", "dingin"]},
])

print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk': 5


In [3]:
satu_produk = koleksi_produk.find_one({"kategori": "Minuman"})
print("Satu produk kategori Minuman:", satu_produk)

print()

print("Seluruh produk kategori Minuman:")
for p in koleksi_produk.find({"kategori": "Minuman"}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

print()

print("Produk dengan harga > 15000:")
for p in koleksi_produk.find({"harga": {"$gt": 15000}}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

Satu produk kategori Minuman: {'_id': ObjectId('6aeb42b3c5bf761b3683c887'), 'nama_produk': 'Kopi Susu', 'kategori': 'Minuman', 'harga': 18000, 'stok': 120, 'tags': ['best-seller', 'dingin', 'panas', 'hangat']}

Seluruh produk kategori Minuman:
  - Kopi Susu (Rp18,000)
  - Es Teh (Rp10,000)
  - Cappuccino (Rp25,000)

Produk dengan harga > 15000:
  - Kopi Susu (Rp18,000)
  - Kentang Goreng (Rp200,000)
  - Cappuccino (Rp25,000)


In [9]:
koleksi_produk.update_one(
    {"nama_produk": "Kentang Goreng"},
    {"$set": {"harga": 250000}}   
)
koleksi_produk.update_many(
    {"kategori": "Minuman"},
    {"$inc": {"stok": -5}}  
)

print("Setelah update:")
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "harga": 1, "stok": 1}):
    print(f"  - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}")

Setelah update:
  - Kopi Susu: harga=Rp18,000, stok=105
  - Roti Bakar: harga=Rp3,000, stok=10
  - Es Teh: harga=Rp10,000, stok=10
  - Kentang Goreng: harga=Rp250,000, stok=5
  - Cappuccino: harga=Rp25,000, stok=65


In [10]:
koleksi_produk.delete_one({"nama_produk": "Cappucino"})

print("Jumlah dokumen tersisa:", koleksi_produk.count_documents({}))
print("Daftar produk tersisa:", [p["nama_produk"] for p in koleksi_produk.find({}, {"nama_produk": 1, "_id": 0})])

Jumlah dokumen tersisa: 5
Daftar produk tersisa: ['Kopi Susu', 'Roti Bakar', 'Es Teh', 'Kentang Goreng', 'Cappuccino']


In [11]:
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

dokumen_mongo = list(koleksi_produk.find({}))

df_pandas = pd.DataFrame(dokumen_mongo)

df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

df_spark_produk = spark.createDataFrame(df_pandas)

df_spark_produk.printSchema()
df_spark_produk.show()

root
 |-- _id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tags: string (nullable = true)

+--------------------+--------------+--------+------+----+--------------------+
|                 _id|   nama_produk|kategori| harga|stok|                tags|
+--------------------+--------------+--------+------+----+--------------------+
|6aeb42b3c5bf761b3...|     Kopi Susu| Minuman| 18000| 105|best-seller, ding...|
|6aeb42b3c5bf761b3...|    Roti Bakar| Makanan|  3000|  10|      smokey, hangat|
|6aeb42b3c5bf761b3...|        Es Teh| Minuman| 10000|  10|        manis, Tidak|
|6aeb42b3c5bf761b3...|Kentang Goreng| Makanan|250000|   5|                asin|
|6aeb42b3c5bf761b3...|    Cappuccino| Minuman| 25000|  65|hangat, best-sell...|
+--------------------+--------------+--------+------+----+--------------------+



In [12]:
from pyspark.sql.functions import avg, count

# Contoh analisis sederhana: rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_produk").alias("jumlah_produk")
).show()

spark.stop()
print("SparkSession ditutup.")

+--------+------------------+-------------+
|kategori|   rata_rata_harga|jumlah_produk|
+--------+------------------+-------------+
| Minuman|17666.666666666668|            3|
| Makanan|          126500.0|            2|
+--------+------------------+-------------+

SparkSession ditutup.


LATIHAN SOAL
**Soal 1.** Sisipkan (`insert_many`) lima dokumen baru ke collection `produk` dengan kategori bebas (boleh kategori baru selain Makanan/Minuman, misalnya "Snack").

In [16]:
# JAWABAN 1

koleksi_produk.insert_many([
    {"nama_produk": "Mie Ayam", "kategori": "Makanan", "harga": 30000, "stok": 30, "tags": ["ceker", "pangsit", "original"]},
    {"nama_produk": "Brownies", "kategori": "Snack", "harga": 25000, "stok": 20, "tags": ["sweet", "salt", "bitter"]},
    {"nama_produk": "Teh Sisri", "kategori": "Minuman", "harga": 8000, "stok": 15, "tags": ["extra", "blackcurrent", "apple", "lemon tea", "melati", "gula batu"]},
    {"nama_produk": "Nasi Goreng", "kategori": "Makanan", "harga": 22000, "stok": 12, "tags": ["tidak pedas", "pedas sedang", "pedas banget", "request cabai"]},
    {"nama_produk": "kerupuk", "kategori": "Snack", "harga": 8000, "stok": 25, "tags": ["udang", "singkog", "rambak", "jengkol"]}
])

print("Lima dokumen berhasil ditambahkan.")
print("Jumlah seluruh produk:", koleksi_produk.count_documents({}))

Lima dokumen berhasil ditambahkan.
Jumlah seluruh produk: 10


In [18]:
### Soal 2. Menampilkan produk dengan harga di bawah 20000

for p in koleksi_produk.find({"harga": {"$lt": 20000}}, {"_id": 0}):
    print(p)

{'nama_produk': 'Kopi Susu', 'kategori': 'Minuman', 'harga': 18000, 'stok': 105, 'tags': ['best-seller', 'dingin', 'panas', 'hangat']}
{'nama_produk': 'Roti Bakar', 'kategori': 'Makanan', 'harga': 3000, 'stok': 10, 'tags': ['smokey', 'hangat']}
{'nama_produk': 'Es Teh', 'kategori': 'Minuman', 'harga': 10000, 'stok': 10, 'tags': ['manis', 'Tidak']}
{'nama_produk': 'Teh Sisri', 'kategori': 'Minuman', 'harga': 8000, 'stok': 15, 'tags': ['extra', 'blackcurrent', 'apple', 'lemon tea', 'melati', 'gula batu']}
{'nama_produk': 'kerupuk', 'kategori': 'Snack', 'harga': 8000, 'stok': 25, 'tags': ['udang', 'singkog', 'rambak', 'jengkol']}


In [20]:
### Field diskon

hasil_update = koleksi_produk.update_many(
    {},
    {"$set": {"diskon": 0}}
)

print("Jumlah dokumen yang diperbarui:", hasil_update.modified_count)
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "diskon": 1}):
    print(p)

Jumlah dokumen yang diperbarui: 0
{'nama_produk': 'Kopi Susu', 'diskon': 0}
{'nama_produk': 'Roti Bakar', 'diskon': 0}
{'nama_produk': 'Es Teh', 'diskon': 0}
{'nama_produk': 'Kentang Goreng', 'diskon': 0}
{'nama_produk': 'Cappuccino', 'diskon': 0}
{'nama_produk': 'Mie Ayam', 'diskon': 0}
{'nama_produk': 'Brownies', 'diskon': 0}
{'nama_produk': 'Teh Sisri', 'diskon': 0}
{'nama_produk': 'Nasi Goreng', 'diskon': 0}
{'nama_produk': 'kerupuk', 'diskon': 0}


### soal nomor 4

MongoDB sangat cocok untuk menyimpan catatan dari berbagai alat (sensor) IoT. Setiap alat punya fungsi yang beda-beda. Contohnya, alat pengukur suhu mencatat derajat panas, sedangkan alat pengukur kelembapan mencatat kebasahan udara. Karena MongoDB sifatnya bebas dan tidak kaku, setiap alat bisa langsung menyimpan catatannya sendiri-sendiri tanpa dipaksa memiliki format atau kolom yang sama.

In [ ]:
##### TUGAS MANDIRI

In [23]:
##### SOAL A

from pymongo import MongoClient

client_kopi = MongoClient("mongodb://localhost:27017/")
db_kopi = client_kopi["kopi_nusantara"]
koleksi_menu = db_kopi["menu"]

# Menghapus data sebelumnya agar data tidak berulang saat cell dijalankan kembali
koleksi_menu.delete_many({})

data_menu = [
    {"nama_menu": "Long Black", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 15, "waktu_seduh_menit": 3},
    {"nama_menu": "Macchiato", "harga": 24000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 12, "waktu_seduh_menit": 2},
    {"nama_menu": "Flat White", "harga": 26000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 20, "rekomendasi_barista": True, "waktu_seduh_menit": 3},
    {"nama_menu": "Kopi Joss", "harga": 16000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 35,  "waktu_seduh_menit": 4},
    {"nama_menu": "Avocado Coffee", "harga": 29000, "kategori": "Kopi", "tingkat_kepahitan": 1, "stok": 10, "rekomendasi_barista": True, "waktu_seduh_menit": 5},
    {"nama_menu": "Mochaccino", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 14, "rekomendasi_barista": True, "waktu_seduh_menit": 4},
    {"nama_menu": "Cold Brew Mint", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 8, "waktu_seduh_menit": 1},
    {"nama_menu": "Affogato Vanilla", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 12, "rekomendasi_barista": False, "waktu_seduh_menit": 2},
    {"nama_menu": "Cireng Bumbu Rujak", "harga": 14000, "kategori": "Makanan", "pedas_level": 2, "stok": 30},
    {"nama_menu": "Singkong Goreng", "harga": 13000, "kategori": "Makanan", "pedas_level": 0, "stok": 22},
    {"nama_menu": "Spaghetti Jowo", "harga": 23000, "kategori": "Makanan", "pedas_level": 1, "stok": 8},
    {"nama_menu": "Sandwich Bakar", "harga": 19000, "kategori": "Makanan", "pedas_level": 0, "stok": 11},
    {"nama_menu": "Rice Bowl Sambal Matah", "harga": 32000, "kategori": "Makanan", "pedas_level": 4, "stok": 15, "rekomendasi_barista": True},
    {"nama_menu": "Jamur Crispy", "harga": 16000, "kategori": "Makanan", "pedas_level": 1, "stok": 18},
    {"nama_menu": "Croissant Cokelat", "harga": 21000, "kategori": "Makanan", "pedas_level": 0, "stok": 7}
]

koleksi_menu.insert_many(data_menu)
print("Jumlah menu berhasil ditambahkan:", koleksi_menu.count_documents({}))
print("Database:", db_kopi.name)
print("Collection:", koleksi_menu.name) 


Jumlah menu berhasil ditambahkan: 15
Database: kopi_nusantara
Collection: menu


In [24]:
### SOAL B KOPI DENGAN KEPAHITAN DI ATAS 3

for menu in koleksi_menu.find(
    {"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}},
    {"_id": 0}
):
    print(menu)

{'nama_menu': 'Long Black', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 15, 'waktu_seduh_menit': 3}
{'nama_menu': 'Kopi Joss', 'harga': 16000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 35, 'waktu_seduh_menit': 4}


In [25]:
### SOAL B MENU HARGA ANTARA 15000 DAN 30000

for menu in koleksi_menu.find(
    {"harga": {"$gte": 15000, "$lte": 30000}},
    {"_id": 0}
):
    print(menu)

{'nama_menu': 'Long Black', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 15, 'waktu_seduh_menit': 3}
{'nama_menu': 'Macchiato', 'harga': 24000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 12, 'waktu_seduh_menit': 2}
{'nama_menu': 'Flat White', 'harga': 26000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 20, 'rekomendasi_barista': True, 'waktu_seduh_menit': 3}
{'nama_menu': 'Kopi Joss', 'harga': 16000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 35, 'waktu_seduh_menit': 4}
{'nama_menu': 'Avocado Coffee', 'harga': 29000, 'kategori': 'Kopi', 'tingkat_kepahitan': 1, 'stok': 10, 'rekomendasi_barista': True, 'waktu_seduh_menit': 5}
{'nama_menu': 'Mochaccino', 'harga': 28000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 14, 'rekomendasi_barista': True, 'waktu_seduh_menit': 4}
{'nama_menu': 'Cold Brew Mint', 'harga': 27000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 8, 'waktu_seduh_menit': 1}
{'nama_menu': 'Affogato Vanilla', 'harga

In [26]:
### SOAL B MENU YANG TIDAK ADA FIELD PEDAS LEVEL

for menu in koleksi_menu.find(
    {"pedas_level": {"$exists": False}},
    {"_id": 0}
):
    print(menu)

{'nama_menu': 'Long Black', 'harga': 22000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 15, 'waktu_seduh_menit': 3}
{'nama_menu': 'Macchiato', 'harga': 24000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 12, 'waktu_seduh_menit': 2}
{'nama_menu': 'Flat White', 'harga': 26000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 20, 'rekomendasi_barista': True, 'waktu_seduh_menit': 3}
{'nama_menu': 'Kopi Joss', 'harga': 16000, 'kategori': 'Kopi', 'tingkat_kepahitan': 4, 'stok': 35, 'waktu_seduh_menit': 4}
{'nama_menu': 'Avocado Coffee', 'harga': 29000, 'kategori': 'Kopi', 'tingkat_kepahitan': 1, 'stok': 10, 'rekomendasi_barista': True, 'waktu_seduh_menit': 5}
{'nama_menu': 'Mochaccino', 'harga': 28000, 'kategori': 'Kopi', 'tingkat_kepahitan': 2, 'stok': 14, 'rekomendasi_barista': True, 'waktu_seduh_menit': 4}
{'nama_menu': 'Cold Brew Mint', 'harga': 27000, 'kategori': 'Kopi', 'tingkat_kepahitan': 3, 'stok': 8, 'waktu_seduh_menit': 1}
{'nama_menu': 'Affogato Vanilla', 'harga

In [28]:
### SOAL C UPDATE

hasil_status = koleksi_menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)
print("Jumlah dokumen yang diberi status:", hasil_status.modified_count)

koleksi_menu.update_one(
    {"nama_menu": "Avocado Coffee"},
    {"$set": {"status": "Habis"}}
)
koleksi_menu.update_one(
    {"nama_menu": "Spaghetti Jowo"},
    {"$set": {"status": "Habis"}}
)

print("Status menu setelah diperbarui:")
for menu in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(menu)

Jumlah dokumen yang diberi status: 0
Status menu setelah diperbarui:
{'nama_menu': 'Long Black', 'status': 'Tersedia'}
{'nama_menu': 'Macchiato', 'status': 'Tersedia'}
{'nama_menu': 'Flat White', 'status': 'Tersedia'}
{'nama_menu': 'Kopi Joss', 'status': 'Tersedia'}
{'nama_menu': 'Avocado Coffee', 'status': 'Habis'}
{'nama_menu': 'Mochaccino', 'status': 'Tersedia'}
{'nama_menu': 'Cold Brew Mint', 'status': 'Tersedia'}
{'nama_menu': 'Affogato Vanilla', 'status': 'Tersedia'}
{'nama_menu': 'Cireng Bumbu Rujak', 'status': 'Tersedia'}
{'nama_menu': 'Singkong Goreng', 'status': 'Tersedia'}
{'nama_menu': 'Spaghetti Jowo', 'status': 'Habis'}
{'nama_menu': 'Sandwich Bakar', 'status': 'Tersedia'}
{'nama_menu': 'Rice Bowl Sambal Matah', 'status': 'Tersedia'}
{'nama_menu': 'Jamur Crispy', 'status': 'Tersedia'}
{'nama_menu': 'Croissant Cokelat', 'status': 'Tersedia'}


In [29]:
### SOAL D INTEGRASI KE SPARK

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg
import pandas as pd

# Menggunakan tanda kurung agar aman dari error spasi/garis miring di Jupyter
spark = (SparkSession.builder 
    .appName("Tugas9-KopiNusantara") 
    .master("local[*]") 
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

dokumen_menu = list(koleksi_menu.find({}, {"_id": 0}))
df_pandas_menu = pd.DataFrame(dokumen_menu)

# Menyamakan kolom agar semua dokumen memiliki struktur kolom yang seragam
kolom_menu = [
    "nama_menu", "harga", "kategori", "tingkat_kepahitan",
    "pedas_level", "stok", "waktu_seduh_menit", "rekomendasi_barista", "status"
]
df_pandas_menu = df_pandas_menu.reindex(columns=kolom_menu)

# Mengubah nilai kosong/NaN menjadi None agar terbaca sebagai null oleh Spark
df_pandas_menu = df_pandas_menu.astype(object).where(pd.notna(df_pandas_menu), None)

df_spark_menu = spark.createDataFrame(df_pandas_menu)

# 1. Menampilkan skema DataFrame
df_spark_menu.printSchema()

# Menampilkan isi DataFrame
df_spark_menu.show(truncate=False)

# 2. Menghitung rata-rata harga berdasarkan kategori
df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga")
).show()

spark.stop()
print("SparkSession ditutup.")


root
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- waktu_seduh_menit: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)



+----------------------+-----+--------+-----------------+-----------+----+-----------------+-------------------+--------+
|nama_menu             |harga|kategori|tingkat_kepahitan|pedas_level|stok|waktu_seduh_menit|rekomendasi_barista|status  |
+----------------------+-----+--------+-----------------+-----------+----+-----------------+-------------------+--------+
|Long Black            |22000|Kopi    |4.0              |NULL       |15  |3.0              |NULL               |Tersedia|
|Macchiato             |24000|Kopi    |3.0              |NULL       |12  |2.0              |NULL               |Tersedia|
|Flat White            |26000|Kopi    |2.0              |NULL       |20  |3.0              |true               |Tersedia|
|Kopi Joss             |16000|Kopi    |4.0              |NULL       |35  |4.0              |NULL               |Tersedia|
|Avocado Coffee        |29000|Kopi    |1.0              |NULL       |10  |5.0              |true               |Habis   |
|Mochaccino            |

#### SOAL E KESIMPULAN

### Tantagan yang dihadapi saat memindahkan data dari MongoDB ke Spark adalah bentuk catatannya yang berbeda-beda, seperti Menu Kopi memiliki tingkat_kepahitan, sedangkan menu Makanan ada pedas_level. Akibatnya ada menu yang tidak memiliki kolom seperti menu lainnya. Padahal di Spark wajib memiliki kolom yang sama rata agar datanya bisa rapi berbentuk tabel.

### Solusinya itu dimasukkan dulu data dari MongoDB ke Pandas. DI situ semua kolomnya di samakan memakai perintah reindex(). Kalau ada menu yang tidak memiliki kolom tertentu, kolom itu otomatis dikosongkan saja (bakal diisi None atau null) . Setelah semuanya punya kolom yang sama , barulah data tersebut dimasukkan ke Spark. Dengan ini catatan asli tiap menu tidakk ada yang hilang dan Spark tetap bisa menghitung rata-rata harga makanan atau kopi tanpa terganggu oleh kolom kosong.